In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import cross_val_score

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from IPython.display import display

### Lecture des données

In [2]:
df = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_walmart/WALMART_SALES/walmart_sales.csv")

In [3]:
df.head(1)

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858


### Qualité des données : 
- Outlyers
- Traitement des dates
- Evaluation des valeurs manquantes

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB


In [5]:
df.describe()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
mean,9.866667,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000
max,20.000000,2.771397e+06,1.000000,91.650000,4.193000,226.968844,14.313000


In [6]:
df.isnull().sum()

Store            0
Date            18
Weekly_Sales    14
Holiday_Flag    12
Temperature     18
Fuel_Price      14
CPI             12
Unemployment    15
dtype: int64

Traitement des dates : 

In [7]:
df["Store"].value_counts()

Store
3.0     15
1.0     11
18.0    10
19.0     9
5.0      9
14.0     9
13.0     9
7.0      8
17.0     8
2.0      8
8.0      8
6.0      7
20.0     7
4.0      7
12.0     5
10.0     5
15.0     4
16.0     4
9.0      4
11.0     3
Name: count, dtype: int64

In [8]:
df = df.dropna(subset=["Date"])
df = df.dropna(subset=["Weekly_Sales"])

In [9]:
df["Store"] = df["Store"].astype(int)
store_counts = df["Store"].value_counts()
print(store_counts)

Store
3     10
14     9
13     9
19     8
1      8
18     7
7      7
5      7
6      6
4      6
8      6
2      6
12     5
17     5
20     4
16     4
9      4
15     3
10     3
11     1
Name: count, dtype: int64


In [10]:
mean = df['Unemployment'].mean()
sigma = df['Unemployment'].std()

mask = df['Unemployment'].between(mean - 3 * sigma, mean + 3 * sigma)
df = df[mask]

In [11]:
df = df[df["Store"].isin(store_counts[store_counts >= 2].index)]
print(df["Store"].value_counts())

Store
3     10
13     9
19     8
14     7
7      7
1      7
6      6
4      6
18     6
5      6
8      5
2      5
20     4
16     4
10     3
17     3
9      3
15     2
Name: count, dtype: int64


In [12]:
print(df["Date"].isnull().sum())
print(df["Weekly_Sales"].isnull().sum())

0
0


In [13]:
df['date'] = pd.to_datetime(df['Date'], format = "%d-%m-%Y")
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.dayofweek
df = df.drop(columns=['Date'])

In [14]:
df[['date', 'year', 'month', 'day', 'weekday']].head(1)

,date,year,month,day,weekday
0,2011-02-18,2011,2,18,4


In [15]:
df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)
df["day"] = df["day"].astype(int)
df["weekday"] = df["weekday"].astype(int)

In [16]:
df["Holiday_Flag"] = df["Holiday_Flag"].astype("Int64")

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 101 entries, 0 to 149
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         101 non-null    int64         
 1   Weekly_Sales  101 non-null    float64       
 2   Holiday_Flag  92 non-null     Int64         
 3   Temperature   93 non-null     float64       
 4   Fuel_Price    91 non-null     float64       
 5   CPI           93 non-null     float64       
 6   Unemployment  101 non-null    float64       
 7   date          101 non-null    datetime64[ns]
 8   year          101 non-null    int64         
 9   month         101 non-null    int64         
 10  day           101 non-null    int64         
 11  weekday       101 non-null    int64         
dtypes: Int64(1), datetime64[ns](1), float64(5), int64(5)
memory usage: 10.4 KB


### Split 

In [18]:
X = df.drop(columns=["Weekly_Sales"])
y = df["Weekly_Sales"]

In [19]:
print(X["Store"].value_counts().sort_index())

Store
1      7
2      5
3     10
4      6
5      6
6      6
7      7
8      5
9      3
10     3
13     9
14     7
15     2
16     4
17     3
18     6
19     8
20     4
Name: count, dtype: int64


In [20]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X["Store"])

Le stratify = X["Store"] permet de s'assurer que la répartition des magasins est similaire dans les ensembles d'entraînement et de test. Cela est important pour éviter un biais dans l'évaluation du modèle, car certains magasins pourraient avoir des caractéristiques de vente différentes. En stratifiant par magasin, on garantit que chaque ensemble contient une proportion représentative de chaque magasin, ce qui permet une évaluation plus fiable des performances du modèle sur l'ensemble des magasins.

### Preprocessing des données :

In [21]:
numeric_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment", "year", "month", "day", "weekday"]
categorical_features = ["Store", "Holiday_Flag"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()) 
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [22]:
x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)

In [23]:
lr_model = LinearRegression()

lr_model.fit(x_train, y_train)

y_train_pred = lr_model.predict(x_train)
y_test_pred = lr_model.predict(x_test)
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

print(f"R² train : {r2_train:.4f}")
print(f"R² test : {r2_test:.4f}")
print(f"Difference R² : {r2_train - r2_test:.4f}")
print()
print(f"MAE train : {mae_train:.4f}")
print(f"MAE test : {mae_test:.4f}")
print(f"Difference MAE : {mae_train - mae_test:.4f}")
print()
print(f"RMSE train : {rmse_train:.4f}")
print(f"RMSE test : {rmse_test:.4f}")
print(f"Difference RMSE : {rmse_train - rmse_test:.4f}")


R² train : 0.9779
R² test : 0.8986
Difference R² : 0.0793

MAE train : 79782.4600
MAE test : 148986.5061
Difference MAE : -69204.0461

RMSE train : 101944.9429
RMSE test : 193096.2936
Difference RMSE : -91151.3506


In [24]:
vente_moyenne = y_test.mean()
vente_mediane = y_test.median()

print(f"Vente moyenne test : {vente_moyenne:,.2f}")
print(f"Vente médiane test : {vente_mediane:,.2f}")
print(f"MAE vente moyenne : {mae_test / vente_moyenne:.1%}")
print(f"RMSE vente moyenne : {rmse_test / vente_moyenne:.1%}")

Vente moyenne test : 1,301,360.93
Vente médiane test : 1,539,483.70
MAE vente moyenne : 11.4%
RMSE vente moyenne : 14.8%


### Attention : Le fait de ne pas garder les valeurs au délà de la règle des 3 sigmas réduit fortement le nombre de lignes dans le dataset. Le score en baseline perd 